In [1]:
import json
import time

import cv2
import mediapipe as mp
import tensorflow as tf
import numpy as np
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

I0000 00:00:1795700272.398253   16361 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795700272.403506   16361 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795700273.002491   16361 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795700280.178754   16361 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
MODEL_PATH = "angry_cnn_best.keras"
CONFIG_PATH = "config.json"
LANDMARKER_PATH = "face_landmarker.task"

CAM_INDEX = 0

INPUT_SIZE = (48, 48)
CROP_MARGIN = 0.10
INFER_EVERY = 2
SMOOTH = 0.35
HYSTERESIS = 0.10

EXPLOSION_VIDEO = "explosion.mp4"
EXPLOSION_MASK = "mascara.mkv"

In [3]:
def create_detector():
    options = vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=LANDMARKER_PATH),
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
    )
    return vision.FaceLandmarker.create_from_options(options)

In [4]:
def get_face_box(detector, frame, timestamp_ms):
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect_for_video(mp_image, timestamp_ms)
    if not result.face_landmarks:
        return None
    pts = np.array([(lm.x * w, lm.y * h) for lm in result.face_landmarks[0]])
    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    return x_min, y_min, x_max, y_max

In [5]:
def crop_face(frame, box):
    h, w = frame.shape[:2]
    x_min, y_min, x_max, y_max = box
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    side = max(x_max - x_min, y_max - y_min) * (1 + 2 * CROP_MARGIN)
    x0, x1 = int(max(cx - side / 2, 0)), int(min(cx + side / 2, w))
    y0, y1 = int(max(cy - side / 2, 0)), int(min(cy + side / 2, h))
    if x1 - x0 < 10 or y1 - y0 < 10:
        return None, None
    return frame[y0:y1, x0:x1], (x0, y0, x1, y1)

In [6]:
def predict_angry(model, face_bgr):
    gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, INPUT_SIZE, interpolation=cv2.INTER_AREA)
    x = gray.astype("float32")[None, :, :, None]
    prob = float(model(x, training=False)[0, 0])
    return prob, gray

In [7]:
class Explosion:

    def __init__(self, video_path, mask_path, max_width=480):
        cap_v = cv2.VideoCapture(video_path)
        cap_m = cv2.VideoCapture(mask_path)
        if not cap_v.isOpened() or not cap_m.isOpened():
            raise FileNotFoundError("No se pudo abrir el vídeo o la máscara")

        self.fps = cap_v.get(cv2.CAP_PROP_FPS) or 25.0
        self.frames, self.masks = [], []

        while True:
            ok_v, f = cap_v.read()
            ok_m, m = cap_m.read()
            if not (ok_v and ok_m):
                break
            if f.shape[1] > max_width:
                s = max_width / f.shape[1]
                f = cv2.resize(f, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
            m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)  # OpenCV la entrega con 3 canales
            m = cv2.resize(m, (f.shape[1], f.shape[0]), interpolation=cv2.INTER_AREA)
            self.frames.append(f)
            self.masks.append(m)

        cap_v.release()
        cap_m.release()
        if not self.frames:
            raise ValueError("El vídeo o la máscara no tienen fotogramas")

        self.t_start = None

    @property
    def active(self):
        return self.t_start is not None

    def play(self):
        if not self.active:
            self.t_start = time.perf_counter()

    def draw_fullscreen(self, frame):
        if not self.active:
            return
        idx = int((time.perf_counter() - self.t_start) * self.fps)
        if idx >= len(self.frames):
            self.t_start = None
            return

        h, w = frame.shape[:2]
        fg = cv2.resize(self.frames[idx], (w, h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(self.masks[idx], (w, h), interpolation=cv2.INTER_LINEAR)
        alpha = (mask.astype(np.float32) / 255.0)[:, :, None]
        frame[:] = (fg * alpha + frame * (1.0 - alpha)).astype(np.uint8)

In [8]:
model = tf.keras.models.load_model(MODEL_PATH)
with open(CONFIG_PATH) as f:
    umbral = float(json.load(f)["umbral"])
model(np.zeros((1, *INPUT_SIZE, 1), dtype="float32"), training=False)

detector = create_detector()
v = cv2.VideoCapture(CAM_INDEX)


prob_smooth = None
angry = False

frame_idx = 0
t_0 = time.perf_counter()

explosion = Explosion(EXPLOSION_VIDEO, EXPLOSION_MASK)

while True:
    ok, frame = v.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)

    box = get_face_box(detector, frame, int(time.monotonic() * 1000))
    timestamp = int((time.perf_counter() - t_0) * 1000)

    if box is None:
        prob_smooth = None
        angry = False
    else:
        face, crop_box = crop_face(frame, box)

        if face is not None and frame_idx % INFER_EVERY == 0:
            t0 = time.perf_counter()
            prob, gray = predict_angry(model, face)
            t_infer = (time.perf_counter() - t0) * 1000

            prob_smooth = prob if prob_smooth is None else (
                    prob_smooth + SMOOTH * (prob - prob_smooth))

            if not angry and prob_smooth >= umbral:
                    angry = True
                    explosion.play()
            elif angry and prob_smooth < umbral - HYSTERESIS:
                angry = False

        frame_idx += 1


    h, w = frame.shape[:2]    
    if explosion.active and box is not None:
        cx = (box[0] + box[2]) / 2
        cy = (box[1] + box[3]) / 2
    explosion.draw_fullscreen(frame)
    
    cv2.imshow("Filtro", frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), 27):
        break

v.release()
detector.close()
cv2.destroyAllWindows()


E0000 00:00:1795700286.649905   16361 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1795700288.014656   16530 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795700288.056400   16534 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795700288.091450   16534 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
